# PDC: Verifikasi & Completion Run (Semua Subjek)

Notebook ini memverifikasi kelengkapan PDC untuk semua 15 subjek dan memproses ulang bila ada yang kurang.

## Status dari investigasi:
- ✅ **PDC**: Subject 15 sudah ada di folder output (3 sesi × 15 trial × 6 band = 270 file)
- ✅ `pdc_metadata.json` mencatat 675 total trial (15 subjek)

## Yang dilakukan notebook ini:
1. **Verifikasi** semua 15 subjek — cek jumlah file per sesi (harus 90 file per sesi)
2. **Proses ulang** jika ada sesi yang tidak lengkap
3. **Update metadata** jika ada perubahan

## Format file PDC per trial:
- `pdc_delta_trial_XX.npy`
- `pdc_theta_trial_XX.npy`
- `pdc_alpha_trial_XX.npy`
- `pdc_beta_trial_XX.npy`
- `pdc_gamma_trial_XX.npy`
- `pdc_broadband_trial_XX.npy`
→ **6 band × 15 trial = 90 file per sesi**

## Cell 1: Import Libraries

In [ ]:
import numpy as np
import os
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')

## Cell 2: Konfigurasi

In [ ]:
# ============================================================
# KONFIGURASI
# ============================================================
INPUT_DIR   = r"D:\Skripsi\new_data\00_preprocessing\output\preprocessed_eeg"
OUTPUT_DIR  = r"D:\Skripsi\new_data\02_pdc\output\pdc_matrices"
FIGURES_DIR = r"D:\Skripsi\new_data\02_pdc\output\figures"
META_PATH   = os.path.join(OUTPUT_DIR, 'pdc_metadata.json')

# Parameter PDC
TARGET_FS   = 100
PDC_MAX_ORDER = 15
PDC_MIN_ORDER = 3
PDC_FREQUENCY_BANDS = {
    'delta'    : (0.5,  4),
    'theta'    : (4,    8),
    'alpha'    : (8,   13),
    'beta'     : (13,  30),
    'gamma'    : (30,  45),
}

TRIAL_LABELS = [1, 0, -1, -1, 0, 1, -1, 0, 1, 1, 0, -1, 0, 1, -1]
EMOTION_MAP  = {-1: 'negative', 0: 'neutral', 1: 'positive'}

# ROI Channels untuk PDC (30 channel)
ROI_CHANNELS = [
    'Fp1','Fp2','AF3','AF4',
    'F3','F4','Fz','F7','F8','FC1','FC2','FCz',
    'FT7','FT8','FC5','FC6','FC3','FC4',
    'T7','T8','TP7','TP8',
    'C3','C4','Cz','C1','C2',
    'P3','P4','Pz','CP1','CP2',
]
PDC_N_CHANNELS = len(ROI_CHANNELS)  # 32 channels

# ALL 62 channel names (untuk index mapping)
CHANNEL_NAMES_62 = [
    'Fp1','Fpz','Fp2','AF3','AF4','F7','F5','F3','F1','Fz',
    'F2','F4','F6','F8','FT7','FC5','FC3','FC1','FCz','FC2',
    'FC4','FC6','FT8','T7','C5','C3','C1','Cz','C2','C4',
    'C6','T8','TP7','CP5','CP3','CP1','CPz','CP2','CP4','CP6',
    'TP8','P7','P5','P3','P1','Pz','P2','P4','P6','P8',
    'PO7','PO5','PO3','POz','PO4','PO6','PO8','CB1','O1','Oz',
    'O2','CB2'
]
ROI_INDICES = [i for i, ch in enumerate(CHANNEL_NAMES_62) if ch in ROI_CHANNELS]

EXPECTED_FILES_PER_SESSION = 6 * 15  # 6 band × 15 trial = 90

print('=' * 55)
print('  PDC VERIFICATION & COMPLETION RUN')
print('=' * 55)
print(f'  Input : {INPUT_DIR}')
print(f'  Output: {OUTPUT_DIR}')
print(f'  PDC channels: {PDC_N_CHANNELS} ROI channels')
print(f'  Bands : {list(PDC_FREQUENCY_BANDS.keys())} + broadband')
print(f'  Expected files/sesi: {EXPECTED_FILES_PER_SESSION}')

## Cell 3: Fungsi PDC (identik dengan notebook utama)

In [ ]:
# ============================================================
# FUNGSI PDC
# ============================================================

def fit_mvar(data, max_order=None, min_order=None):
    """Fit MVAR model dengan AIC order selection."""
    max_order = max_order or PDC_MAX_ORDER
    min_order = min_order or PDC_MIN_ORDER
    model = VAR(data)
    try:
        order_result = model.select_order(maxlags=max_order)
        optimal_order = max(order_result.aic, min_order)
    except:
        optimal_order = 5

    model_fitted = model.fit(maxlags=optimal_order)
    n_ch = data.shape[1]
    coefficients = np.zeros((optimal_order, n_ch, n_ch))
    for lag in range(optimal_order):
        start, end = lag * n_ch, (lag + 1) * n_ch
        if end <= model_fitted.params.shape[0]:
            coefficients[lag] = model_fitted.params[start:end].T
    return model_fitted, optimal_order, coefficients


def compute_pdc(coefficients, freqs, fs):
    """Compute PDC dari koefisien MVAR."""
    order, n_ch, _ = coefficients.shape
    pdc = np.zeros((len(freqs), n_ch, n_ch))
    for f_idx, freq in enumerate(freqs):
        A_f = np.eye(n_ch, dtype=complex)
        for lag in range(1, order + 1):
            if lag - 1 < coefficients.shape[0]:
                A_f -= coefficients[lag - 1] * np.exp(-2j * np.pi * freq * lag / fs)
        for j in range(n_ch):
            denom = np.sqrt(np.sum(np.abs(A_f[:, j]) ** 2))
            if denom > 0:
                pdc[f_idx, :, j] = np.abs(A_f[:, j]) / denom
    return pdc


def average_over_bands(pdc, freqs, bands=None):
    """Average PDC per band frekuensi."""
    bands = bands or PDC_FREQUENCY_BANDS
    band_pdc = {}
    for band_name, (fmin, fmax) in bands.items():
        mask = (freqs >= fmin) & (freqs <= fmax)
        band_pdc[band_name] = np.mean(pdc[mask], axis=0) if mask.any() else np.zeros((pdc.shape[1], pdc.shape[2]))
    band_pdc['broadband'] = np.mean(pdc, axis=0)
    return band_pdc


def analyze_trial_pdc(eeg_data, fs=None, max_order=None):
    """Full PDC pipeline untuk 1 trial."""
    fs = fs or TARGET_FS
    # Pilih ROI channels
    roi_data = eeg_data[ROI_INDICES, :].T  # (time, roi_channels)
    # Fit MVAR
    _, optimal_order, coefficients = fit_mvar(roi_data, max_order)
    # Compute PDC
    freqs    = np.linspace(0.5, fs / 2, 200)
    pdc_full = compute_pdc(coefficients, freqs, fs)
    # Average per band
    band_pdc = average_over_bands(pdc_full, freqs)
    return band_pdc, optimal_order


print('Fungsi PDC didefinisikan ✅')
print(f'ROI indices (dari 62 channel): {len(ROI_INDICES)} channel')
print(f'ROI channels: {ROI_CHANNELS}')

## Cell 4: Audit Kelengkapan Semua Subjek

In [ ]:
# ============================================================
# AUDIT — Cek kelengkapan file per sesi per subjek
# ============================================================

print('=' * 70)
print('  AUDIT KELENGKAPAN PDC — SEMUA 15 SUBJEK')
print('=' * 70)

incomplete_sessions = []   # (subj_dir, sess_dir) yang belum lengkap
total_ok = 0
total_incomplete = 0

for subj_dir in sorted(os.listdir(INPUT_DIR)):
    subj_input_path  = os.path.join(INPUT_DIR,  subj_dir)
    subj_output_path = os.path.join(OUTPUT_DIR, subj_dir)

    if not os.path.isdir(subj_input_path):
        continue

    subj_sessions = sorted(os.listdir(subj_input_path))
    subj_ok = 0
    subj_incomplete = 0

    for sess_dir in subj_sessions:
        if not os.path.isdir(os.path.join(subj_input_path, sess_dir)):
            continue

        sess_output = os.path.join(subj_output_path, sess_dir)

        if os.path.exists(sess_output):
            npy_files = [f for f in os.listdir(sess_output) if f.endswith('.npy')]
            n_files   = len(npy_files)
        else:
            n_files = 0

        if n_files == EXPECTED_FILES_PER_SESSION:
            status = '✅'
            subj_ok += 1
            total_ok += 1
        else:
            missing = EXPECTED_FILES_PER_SESSION - n_files
            status  = f'❌ ({n_files}/{EXPECTED_FILES_PER_SESSION}, kurang {missing})'
            subj_incomplete += 1
            total_incomplete += 1
            incomplete_sessions.append((subj_dir, sess_dir))

        print(f'  {subj_dir}/{sess_dir}: {status}')

print()
print('=' * 70)
print(f'  Sesi OK      : {total_ok}')
print(f'  Sesi kurang  : {total_incomplete}')

if total_incomplete == 0:
    print()
    print('  🎉 SEMUA LENGKAP! Tidak ada yang perlu diproses ulang.')
    print('     Lanjut ke Cell 6 untuk update metadata.')
else:
    print()
    print('  Sesi yang perlu diproses:')
    for subj_dir, sess_dir in incomplete_sessions:
        print(f'    - {subj_dir}/{sess_dir}')
    print()
    print(f'  Estimasi waktu: {total_incomplete} sesi × 15 trial × ~14s = '
          f'~{total_incomplete * 15 * 14 / 60:.0f} menit')

## Cell 5: Proses Ulang Sesi yang Kurang

> Jika semua ✅ dari Cell 4, cell ini akan langsung selesai tanpa kerja tambahan.

In [ ]:
# ============================================================
# COMPLETION PROCESSING — Hanya sesi yang tidak lengkap
# ============================================================

if not incomplete_sessions:
    print('✅ Tidak ada yang perlu diproses. PDC sudah lengkap untuk semua subjek.')
else:
    print(f'⏳ Memproses {len(incomplete_sessions)} sesi yang belum lengkap...')
    print()

    new_meta_updates = {}
    start_all = time.time()

    for subj_dir, sess_dir in incomplete_sessions:
        sess_input_path  = os.path.join(INPUT_DIR, subj_dir, sess_dir)
        sess_output_path = os.path.join(OUTPUT_DIR, subj_dir, sess_dir)
        os.makedirs(sess_output_path, exist_ok=True)

        trial_files = sorted([f for f in os.listdir(sess_input_path)
                               if f.endswith('.npy')])

        sess_orders = []
        sess_trials = 0
        start_sess  = time.time()

        for tf in trial_files:
            trial_idx = int(tf.split('_')[1].split('.')[0])
            label     = TRIAL_LABELS[trial_idx - 1]
            emotion   = EMOTION_MAP[label]

            # Cek apakah trial ini sudah ada (partial completion)
            delta_path = os.path.join(sess_output_path,
                                       f'pdc_delta_trial_{trial_idx:02d}.npy')
            broadband_path = os.path.join(sess_output_path,
                                           f'pdc_broadband_trial_{trial_idx:02d}.npy')
            already_done = os.path.exists(delta_path) and os.path.exists(broadband_path)

            if already_done:
                print(f'    SKIP {subj_dir}/{sess_dir}/trial_{trial_idx:02d} ({emotion})')
                continue

            try:
                eeg = np.load(os.path.join(sess_input_path, tf))
                t0  = time.time()
                band_pdc, optimal_order = analyze_trial_pdc(eeg)
                t1  = time.time()

                # Simpan semua band
                for band_name, pdc_mat in band_pdc.items():
                    out_path = os.path.join(
                        sess_output_path,
                        f'pdc_{band_name}_trial_{trial_idx:02d}.npy')
                    np.save(out_path, pdc_mat)

                sess_orders.append(optimal_order)
                sess_trials += 1
                print(f'    DONE {subj_dir}/{sess_dir}/trial_{trial_idx:02d} '
                      f'({emotion}) — order={optimal_order}, {t1-t0:.1f}s')

            except Exception as e:
                print(f'    ERROR {subj_dir}/{sess_dir}/trial_{trial_idx:02d} — {e}')

        sess_time = time.time() - start_sess
        if subj_dir not in new_meta_updates:
            new_meta_updates[subj_dir] = {}
        new_meta_updates[subj_dir][sess_dir] = {
            'n_trials'   : 15,
            'mean_order' : float(np.mean(sess_orders)) if sess_orders else 14.5,
            'time_s'     : round(sess_time, 1),
        }
        print(f'  → {subj_dir}/{sess_dir} selesai: {sess_trials} trial baru '
              f'dalam {sess_time:.0f}s')
        print()

    total_time = time.time() - start_all
    print(f'Selesai! Total waktu: {total_time/60:.1f} menit')
    print(f'Meta updates: {new_meta_updates}')

## Cell 6: Update & Verifikasi pdc_metadata.json

In [ ]:
# ============================================================
# UPDATE METADATA
# ============================================================

with open(META_PATH) as f:
    meta = json.load(f)

print(f'Metadata sebelum: {len(meta["subjects"])} subjek, {meta["total_trials"]} trials')

# Hitung ulang total dari file output aktual
total_trials_actual = 0
all_orders_actual   = []

for subj_dir in sorted(os.listdir(OUTPUT_DIR)):
    subj_out = os.path.join(OUTPUT_DIR, subj_dir)
    if not os.path.isdir(subj_out):
        continue

    if subj_dir not in meta['subjects']:
        meta['subjects'][subj_dir] = {}

    for sess_dir in sorted(os.listdir(subj_out)):
        sess_out = os.path.join(subj_out, sess_dir)
        if not os.path.isdir(sess_out):
            continue

        npy_count = len([f for f in os.listdir(sess_out) if f.endswith('.npy')])
        n_trials  = npy_count // 6   # 6 band per trial
        total_trials_actual += n_trials

        if sess_dir not in meta['subjects'][subj_dir]:
            meta['subjects'][subj_dir][sess_dir] = {
                'n_trials'   : n_trials,
                'mean_order' : 14.5,
                'time_s'     : 0.0,
            }
        else:
            meta['subjects'][subj_dir][sess_dir]['n_trials'] = n_trials

meta['total_trials'] = total_trials_actual

with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Metadata sesudah: {len(meta["subjects"])} subjek, {meta["total_trials"]} trials')
print(f'Target: 15 subjek, 675 trials')

if meta['total_trials'] == 675 and len(meta['subjects']) == 15:
    print()
    print('🎉 METADATA SEMPURNA! 15 subjek × 675 trial ✅')
else:
    diff = 675 - meta['total_trials']
    print(f'⚠️  Masih kurang {diff} trial!')

## Cell 7: Audit Final & Visualisasi

In [ ]:
# ============================================================
# AUDIT FINAL
# ============================================================

print('=' * 70)
print('  AUDIT FINAL — STATUS SEMUA 15 SUBJEK')
print('=' * 70)

all_complete = True
subjects_data = []

for subj_dir in sorted(os.listdir(INPUT_DIR)):
    subj_input  = os.path.join(INPUT_DIR, subj_dir)
    subj_output = os.path.join(OUTPUT_DIR, subj_dir)
    if not os.path.isdir(subj_input):
        continue

    sessions = [d for d in sorted(os.listdir(subj_input))
                if os.path.isdir(os.path.join(subj_input, d))]
    subj_ok  = True
    subj_files = 0

    for sess_dir in sessions:
        sess_out = os.path.join(subj_output, sess_dir)
        if os.path.exists(sess_out):
            n = len([f for f in os.listdir(sess_out) if f.endswith('.npy')])
        else:
            n = 0
        subj_files += n
        if n != EXPECTED_FILES_PER_SESSION:
            subj_ok = False

    total_expected = len(sessions) * EXPECTED_FILES_PER_SESSION
    status = '✅ LENGKAP' if subj_ok else f'❌ {subj_files}/{total_expected}'
    print(f'  {subj_dir}: {status}')

    subjects_data.append(subj_dir)
    if not subj_ok:
        all_complete = False

print()
if all_complete:
    print('  🎉 SEMUA 15 SUBJEK LENGKAP!')
else:
    print('  ⚠️  Ada yang belum lengkap — jalankan Cell 5 lagi')

# ============================================================
# VISUALISASI — Density PDC per band per subjek
# ============================================================
print()
print('Membuat visualisasi density PDC per subjek...')

BANDS_TO_PLOT = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'broadband']
subject_list  = sorted([d for d in os.listdir(INPUT_DIR)
                          if os.path.isdir(os.path.join(INPUT_DIR, d))])
n_subj = len(subject_list)

# Sample: ambil trial_01 sesi_pertama per subjek untuk setiap band
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('PDC Band Matrices — Subjek 15 (Sesi 1, Trial 1)', fontsize=14)
axes = axes.flatten()

for ax_idx, band in enumerate(BANDS_TO_PLOT):
    ax = axes[ax_idx]
    subj15_sess1 = os.path.join(
        OUTPUT_DIR, 'subject_15', 'session_20130709')
    pdc_path = os.path.join(subj15_sess1, f'pdc_{band}_trial_01.npy')

    if os.path.exists(pdc_path):
        mat = np.load(pdc_path)
        sns.heatmap(mat, ax=ax, cmap='viridis',
                    xticklabels=False, yticklabels=False,
                    vmin=0, vmax=1, cbar_kws={'shrink': 0.8})
        mean_pdc = mat[mat > 0].mean() if mat.any() else 0
        ax.set_title(f'{band.capitalize()} band\nmean PDC={mean_pdc:.3f}',
                     fontsize=11)
    else:
        ax.text(0.5, 0.5, f'{band}\nFile tidak ditemukan',
                ha='center', va='center', transform=ax.transAxes, color='red')

    ax.set_xlabel('Source')
    ax.set_ylabel('Target')

plt.tight_layout()
os.makedirs(FIGURES_DIR, exist_ok=True)
fig_path = os.path.join(FIGURES_DIR, 'subject_15_pdc_bands.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure disimpan: {fig_path}')

---
## ✅ Selesai!

Setelah notebook ini selesai:
- `pdc_matrices/` berisi **15 subjek × 3 sesi × 90 file = 4050 file .npy**
- `pdc_metadata.json` mencakup **15 subjek, total 675 trial**
- Lanjutkan ke pipeline Phase 1 data structuring